In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/thuandao/bank-transactions-dataset-for-fraud-detection/bank_transactions_data_2_augmented_clean_2.csv


# Problem Statement

Banks face challenges detecting unusual transactions as digital banking grows. AI and machine learning can analyze transaction patterns to detect anomalies, reduce fraud, and improve risk management.

**Why AI Matters**

- Detect fraud proactively

- Improve risk management

- Enable data-driven decisions

- Enhance customer experience

**Objectives**

- Clean and preprocess transaction data

- Explore patterns via EDA and visualizations

- Engineer features for modeling

- Detect anomalies using Isolation Forest, LOF, OCSVM

- Train multiple ML models (LogReg, DT, RF, GB, XGB, KNN, AdaBoost)

- Compare model performance metrics

- Explain predictions using SHAP

- Visualize patterns with PCA, t-SNE, and clustering

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.ensemble import IsolationForest, RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.neighbors import LocalOutlierFactor, KNeighborsClassifier
from sklearn.svm import OneClassSVM
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.cluster import KMeans, DBSCAN
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

#  Load Dataset

In [3]:

df = pd.read_csv("/kaggle/input/datasets/thuandao/bank-transactions-dataset-for-fraud-detection/bank_transactions_data_2_augmented_clean_2.csv")
print("Original shape:", df.shape)

Original shape: (50000, 15)


#  Date Features

In [4]:

df["TransactionDate"] = pd.to_datetime(df["TransactionDate"], errors='coerce', infer_datetime_format=True)
df = df.dropna(subset=["TransactionDate"])
df["Year"] = df["TransactionDate"].dt.year
df["Month"] = df["TransactionDate"].dt.month
df["Day"] = df["TransactionDate"].dt.day
df["Hour"] = df["TransactionDate"].dt.hour
df["Weekday"] = df["TransactionDate"].dt.weekday
df.drop("TransactionDate", axis=1, inplace=True)

#  Feature Engineering

In [5]:


account_ids = df['AccountID']
df['TransactionAmount_log'] = np.log1p(df['TransactionAmount'])
df['AccountAvgTransaction'] = df.groupby(account_ids)['TransactionAmount'].transform('mean')
df['AccountTransactionCount'] = df.groupby(account_ids)['TransactionAmount'].transform('count')

df['Channel_Occupation'] = df['Channel'].astype(str) + "_" + df['CustomerOccupation'].astype(str)
df['Channel_Occupation'] = LabelEncoder().fit_transform(df['Channel_Occupation'])

def time_of_day(hour):
    if 5 <= hour < 12: return 0
    elif 12 <= hour < 17: return 1
    elif 17 <= hour < 21: return 2
    else: return 3
df['TimeOfDay'] = df['Hour'].apply(time_of_day)
df.drop("AccountID", axis=1, inplace=True)

# Encode remaining categorical columns
for col in df.select_dtypes(include='object').columns:
    df[col] = LabelEncoder().fit_transform(df[col])

#  Scale Features

In [6]:

features = df.drop(['AccountAvgTransaction','AccountTransactionCount'], axis=1, errors='ignore')
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)


# Unsupervised Anomaly Detection

In [7]:

iso = IsolationForest(contamination=0.02, random_state=42)
df['anomaly_iso'] = iso.fit_predict(X_scaled)
df['anomaly_label'] = df['anomaly_iso'].map({1:0, -1:1})
print(" % anomalies from Isolation Forest:", df['anomaly_label'].mean()*100)


 % anomalies from Isolation Forest: 2.0


# Supervised Models

In [8]:

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(
        random_state=42, n_estimators=100, max_depth=3, learning_rate=0.05, subsample=0.8
    ),
    'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='logloss', n_estimators=100, random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5),
    'AdaBoost': AdaBoostClassifier(n_estimators=100, random_state=42)
}

X_train, X_test, y_train, y_test = train_test_split(X_scaled, df['anomaly_label'], test_size=0.2, random_state=42, stratify=df['anomaly_label'])

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    print(classification_report(y_test, y_pred, digits=4))



              precision    recall  f1-score   support

           0     0.9847    0.9967    0.9907      1225
           1     0.6000    0.2400    0.3429        25

    accuracy                         0.9816      1250
   macro avg     0.7923    0.6184    0.6668      1250
weighted avg     0.9770    0.9816    0.9777      1250

              precision    recall  f1-score   support

           0     0.9877    0.9829    0.9853      1225
           1     0.3226    0.4000    0.3571        25

    accuracy                         0.9712      1250
   macro avg     0.6551    0.6914    0.6712      1250
weighted avg     0.9744    0.9712    0.9727      1250

              precision    recall  f1-score   support

           0     0.9839    0.9959    0.9899      1225
           1     0.5000    0.2000    0.2857        25

    accuracy                         0.9800      1250
   macro avg     0.7419    0.5980    0.6378      1250
weighted avg     0.9742    0.9800    0.9758      1250

              preci

# shap 

In [9]:
import shap


explainer = shap.TreeExplainer(rf_model, data=X_train, model_output="raw")  # must be "raw"
shap_values = explainer.shap_values(X_test, check_additivity=False)

# Summary plot
shap.summary_plot(shap_values, X_test, feature_names=features.columns)
# Summary plot
shap.summary_plot(shap_values, X_test, feature_names=features.columns)


NameError: name 'rf_model' is not defined

#  Visualizations

In [ ]:



# Transaction Amount Distribution
plt.figure(figsize=(10,5))
sns.histplot(df['TransactionAmount_log'], bins=50, kde=True)
plt.title("Log Transaction Amount Distribution")
plt.show()

# Transaction Type Counts
plt.figure(figsize=(8,4))
sns.countplot(x='TransactionType', data=df)
plt.title("Transaction Type Count")
plt.show()

# Time-of-Day Transactions
plt.figure(figsize=(8,4))
sns.countplot(x='TimeOfDay', data=df)
plt.title("Transactions by Time of Day")
plt.show()

# Heatmap: Channel vs CustomerOccupation
plt.figure(figsize=(12,6))
ct_matrix = pd.crosstab(df['Channel'], df['CustomerOccupation'])
sns.heatmap(ct_matrix, annot=False, cmap='coolwarm')
plt.title("Channel vs Customer Occupation")
plt.show()

# Anomalies by Hour
plt.figure(figsize=(10,4))
sns.countplot(x='Hour', hue='anomaly_label', data=df)
plt.title("Anomalies by Hour")
plt.show()

# Anomalies by Weekday
plt.figure(figsize=(10,4))
sns.countplot(x='Weekday', hue='anomaly_label', data=df)
plt.title("Anomalies by Weekday")
plt.show()

# PCA & t-SNE Visualizations
pca = PCA(n_components=2)
pca_data = pca.fit_transform(X_scaled)

tsne = TSNE(n_components=2, random_state=42, perplexity=50)
tsne_data = tsne.fit_transform(X_scaled)

def plot_anomaly_scatter(data_2d, labels, title):
    plt.figure(figsize=(8,6))
    plt.scatter(data_2d[:,0], data_2d[:,1], c=labels, cmap='coolwarm', alpha=0.6)
    plt.title(title)
    plt.show()

plot_anomaly_scatter(pca_data, df['anomaly_label'], "PCA - Isolation Forest Anomalies")
plot_anomaly_scatter(tsne_data, df['anomaly_label'], "t-SNE - Isolation Forest Anomalies")